# 🧬 06 — Finding the same object in different orientations

**Audience:** first-year biology major · **Time:** ~3 hours · **Hardware:** Google Colab free CPU · **Framework:** NumPy · **Prerequisites:** Lessons 02–05; arrays and coordinates.

**Learning goals**
- Separate translation from rotation
- Implement an elementary alignment score
- Explain reference bias and the need for controls

**Route:** biological question → concept → runnable toy model → interpretation → paper bridge. All examples use synthetic data and are for research and education.


## 1. Why this matters

A subtomogram is a small 3D crop around one candidate molecule. Its orientation is usually unknown. If we average crops without aligning them, directional features blur. Alignment searches for a transformation that makes corresponding structures agree. In cryo-ET, missing-wedge overlap and crowding can make a high correlation misleading.


## 2. The idea in plain language

A rigid transformation has a rotation `R` and translation `t`. An elementary score is normalized cross-correlation between a transformed crop and reference. Optimization may search all rotations coarsely, then refine near promising candidates. The reference must be tested independently to avoid pulling noisy particles toward a preferred shape.


## 3. Run the experiment

Run the setup cell, then the activity. Before running the activity, predict one quantity or pattern you expect. The numbers and shapes are illustrative; they are not published data.


In [ ]:
# ── Shared setup (runs on free Colab CPU) ──
# Colab includes NumPy. If running elsewhere, install with: %pip install numpy
import numpy as np
SEED = 7
print('NumPy', np.__version__, '| device: CPU | seed:', SEED)


In [ ]:
template=np.zeros((21,21))
template[7:14,9:12]=1
template[12:15,11:17]=0.7
observed=np.roll(np.rot90(template),shift=(3,-2),axis=(0,1))
def corr(a,b):
    a=a-a.mean(); b=b-b.mean()
    return float((a*b).sum()/np.sqrt((a*a).sum()*(b*b).sum()))
best=(-2,None)
for rotation in range(4):
    candidate=np.rot90(observed,rotation)
    for dy in range(-5,6):
        for dx in range(-5,6):
            score=corr(template,np.roll(candidate,(dy,dx),axis=(0,1)))
            if score>best[0]: best=(score,(rotation,dy,dx))
print('Best correlation and transform:',best)
assert best[0]>0.99


## 4. Interpret, then test a change

This tiny 2D search is deliberately exhaustive; real 3D rotation searches are much larger and use frequency-space constraints, learned pose proposals, or iterative refinement.

**Change one thing:** Add Gaussian noise and repeat the search 20 times. How often is the best transform correct? What changes if the template has rotational symmetry?


## 5. Bridge to research

Compare the alignment strategies in [2012 fast volumetric matching](https://doi.org/10.1016/j.jsb.2012.02.014), [2018 fast alignment plus maximum likelihood](https://doi.org/10.1093/bioinformatics/bty267), [Gum-Net](https://openaccess.thecvf.com/content_CVPR_2020/html/Zeng_Gum-Net_Unsupervised_Geometric_Matching_for_Fast_and_Accurate_3D_Subtomogram_CVPR_2020_paper.html), and [BOE-ViT](https://xulabs.github.io/publication/).

Write a four-line lab note: input, output, key assumption, and one possible failure. In the linked paper, identify the corresponding real data and the evidence for the authors’ main claim.
